# Step 2 — Priority Queue Simulation & Feature Engineering

**What this notebook does:**
1. Assigns priority levels (Normal / High / Critical) to experiments based on machine health, failures, and incident severity
2. Simulates a realistic, non-preemptive priority-aware scheduling queue per instrument
3. Engineers additional temporal, congestion, and reliability features

**Inputs (from Step 1):**
- `../data/raw/workflow_logs.csv`

**Outputs:**
- `../data/enriched/workflow_logs_priority_queue.csv` (enriched with ~10 new features)

**Run order:** Run after Step 1 (notebook 01), before Steps 3–5.

## 1. Setup & Configuration

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import heapq
import time

# =========================================================
# CONFIG
# =========================================================
HEALTH_STRICT_CUTOFF = 0.25    # instrument_health < this → Critical
DURATION_Q_HIGH      = 0.85    # top 15% duration → High priority
SEVERE_INCIDENT_KEYWORDS = ("failure", "contamination", "error", "critical", "abort", "shutdown")
PRIORITY_RANK = {"Low": 1, "Normal": 2, "High": 3, "Critical": 4}

# Paths
DATA_DIR  = Path("..") / "data"
IN_PATH   = DATA_DIR / "raw" / "workflow_logs.csv"
OUT_PATH  = DATA_DIR / "enriched" / "workflow_logs_priority_queue.csv"

assert IN_PATH.exists(), f"Missing {IN_PATH}"

## 2. Load Raw Workflow Data

In [2]:
df = pd.read_csv(IN_PATH)
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")

# Parse types
df["booking_time"] = pd.to_datetime(df["booking_time"], errors="coerce")
df["expected_duration"] = pd.to_numeric(df["expected_duration"], errors="coerce").fillna(0.0)

df.head()

Loaded: 350,000 rows × 14 cols


,experiment_id,experiment_type,instrument_type,instrument_id,scientist_workload,scientist_experience_level,lab_occupancy_level,expected_duration,booking_time,actual_duration,delay,incident_type,machine_failure,instrument_health
0,EXP_000000,Pilot,Incubator,Incubator_04,3,Junior,59,90,2024-01-01 00:18:04,98.944742,8.944742,NaN,0,0.999992
1,EXP_000001,QC,Spectrometer,Spectrometer_04,4,Mid,54,45,2024-01-01 00:44:22,45.895662,0.895662,NaN,0,0.999998
2,EXP_000002,QC,HPLC,HPLC_03,1,Senior,43,45,2024-01-01 01:05:07,45.618095,0.618095,NaN,0,0.999998
3,EXP_000003,QC,Spectrometer,Spectrometer_01,5,Senior,51,45,2024-01-01 01:21:09,63.556023,18.556023,NaN,0,0.999998
4,EXP_000004,Screening,PCR,PCR_00,6,Senior,81,30,2024-01-01 01:42:56,70.318977,40.318977,Resource_Contention,0,0.999999


## 3. Fenwick Tree (Binary Indexed Tree)

A space-efficient data structure for computing prefix sums in **O(log n)**.  
Used by the queue simulation to count how many jobs are "ahead" of each experiment.

In [3]:
class Fenwick:
    """Binary Indexed Tree for O(log n) prefix sums."""
    def __init__(self, n: int):
        self.n = n
        self.bit = np.zeros(n + 1, dtype=np.int32)

    def add(self, i: int, delta: int):
        i += 1
        while i <= self.n:
            self.bit[i] += delta
            i += i & -i

    def sum_prefix(self, i: int) -> int:
        if i < 0:
            return 0
        i += 1
        s = 0
        while i > 0:
            s += self.bit[i]
            i -= i & -i
        return int(s)

## 4. Priority Assignment

Each experiment is assigned a priority level using **leakage-safe** rules:

| Priority | Rule |
|----------|------|
| **Critical** | `machine_failure == 1` OR `instrument_health < 0.25` |
| **High** | Severe incident keyword OR `expected_duration ≥ Q85` |
| **Normal** | Everything else |

These rules use only **pre-execution** information (no target leakage).

In [4]:
def build_priority(df: pd.DataFrame) -> tuple:
    mf = pd.to_numeric(df.get("machine_failure", 0), errors="coerce").fillna(0).astype(int)
    health = pd.to_numeric(df.get("instrument_health", np.nan), errors="coerce")
    crit_health = health.notna() & (health < float(HEALTH_STRICT_CUTOFF))

    dur = pd.to_numeric(df.get("expected_duration", np.nan), errors="coerce")
    dur_thr = float(dur.quantile(DURATION_Q_HIGH)) if dur.notna().any() else np.nan
    high_duration = dur.notna() & (dur >= dur_thr) if np.isfinite(dur_thr) else pd.Series(False, index=df.index)

    incident = df.get("incident_type", "").fillna("").astype(str).str.lower()
    severe_incident = pd.Series(False, index=df.index)
    for kw in SEVERE_INCIDENT_KEYWORDS:
        severe_incident = severe_incident | incident.str.contains(kw, regex=False)

    priority = pd.Series("Normal", index=df.index)
    priority[(mf == 1) | crit_health] = "Critical"
    priority[(priority != "Critical") & (severe_incident | high_duration)] = "High"

    return priority, dur_thr

# Apply
priority, dur_thr = build_priority(df)
df["priority"] = priority
df["priority_rank"] = df["priority"].map(PRIORITY_RANK).astype(int)

print(f"Duration threshold (Q{DURATION_Q_HIGH}): {dur_thr}")
print("\nPriority distribution:")
print(df["priority"].value_counts())

Duration threshold (Q0.85): 90.0

Priority distribution:
priority
Normal      235483
High        100387
Critical     14130
Name: count, dtype: int64


## 5. Priority-Aware Queue Simulation

For each instrument, we simulate a **non-preemptive priority queue**:

- Jobs arrive in chronological order (`booking_time`)
- Higher priority jobs jump ahead in the waiting queue
- A running job is never interrupted (non-preemptive)
- Output: `queue_wait_min` (how long each job waited) and `queue_length` (how many ahead)

**Algorithm complexity:** O(n log n) per instrument using a min-heap + Fenwick tree.

In [5]:
def simulate_schedule_priority_nonpreemptive(booking_ns, dur_min, pr_rank):
    n = len(booking_ns)
    order_arrival = np.argsort(booking_ns, kind="mergesort")
    start_ns = np.empty(n, dtype=np.int64)
    end_ns = np.empty(n, dtype=np.int64)
    sched_idx = np.empty(n, dtype=np.int32)

    heap = []
    t = None
    p = 0
    seq = 0
    out_k = 0

    def push_until(time_ns):
        nonlocal p, seq
        while p < n:
            j = int(order_arrival[p])
            if int(booking_ns[j]) <= time_ns:
                heapq.heappush(heap, (-int(pr_rank[j]), int(booking_ns[j]), seq, j))
                seq += 1
                p += 1
            else:
                break

    while out_k < n:
        if t is None:
            t = int(booking_ns[int(order_arrival[0])])
        push_until(t)
        if not heap:
            jn = int(order_arrival[p])
            t = int(booking_ns[jn])
            push_until(t)
        _, b, _, j = heapq.heappop(heap)
        st = t if t >= b else b
        minutes = float(dur_min[j]) if np.isfinite(dur_min[j]) else 0.0
        et = st + int(round(minutes)) * 60 * 1_000_000_000
        start_ns[j] = st
        end_ns[j] = et
        sched_idx[j] = out_k
        t = et
        out_k += 1
    return start_ns, end_ns, sched_idx


def compute_queue_length_with_events(booking_ns, end_ns, sched_idx):
    n = len(booking_ns)
    arr_order = np.argsort(booking_ns, kind="mergesort")
    end_order = np.argsort(end_ns, kind="mergesort")
    bit = Fenwick(n)
    qlen = np.zeros(n, dtype=np.int32)
    pa = 0
    pe = 0
    for qi in arr_order:
        t = int(booking_ns[int(qi)])
        while pe < n and int(end_ns[int(end_order[pe])]) <= t:
            j = int(end_order[pe])
            bit.add(int(sched_idx[j]), -1)
            pe += 1
        while pa < n and int(booking_ns[int(arr_order[pa])]) <= t:
            j = int(arr_order[pa])
            bit.add(int(sched_idx[j]), +1)
            pa += 1
        si = int(sched_idx[int(qi)])
        qlen[int(qi)] = bit.sum_prefix(si - 1)
    return qlen


def process_one_instrument(g: pd.DataFrame) -> pd.DataFrame:
    booking = g["booking_time"].astype("datetime64[ns]").to_numpy()
    booking_ns = booking.astype("int64")
    dur_min = pd.to_numeric(g["expected_duration"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
    pr = g["priority_rank"].to_numpy(dtype=int)
    start_ns, end_ns, sched_idx = simulate_schedule_priority_nonpreemptive(booking_ns, dur_min, pr)
    wait_min = (start_ns - booking_ns) / (60 * 1_000_000_000)
    wait_min = np.maximum(wait_min, 0.0)
    qlen = compute_queue_length_with_events(booking_ns, end_ns, sched_idx)
    out = g.copy()
    out["queue_wait_min"] = wait_min.astype(float)
    out["queue_length"] = qlen.astype(int)
    return out

print("Queue simulation functions defined.")

Queue simulation functions defined.


## 6. Run Queue Simulation (All Instruments)

In [6]:
if "instrument_id" not in df.columns:
    raise ValueError("Missing instrument_id column")

valid = df["instrument_id"].notna() & df["booking_time"].notna()
df_valid = df.loc[valid].copy()
df_invalid = df.loc[~valid].copy()
df_invalid["queue_wait_min"] = 0.0
df_invalid["queue_length"] = 0

instruments = df_valid["instrument_id"].astype(str).unique().tolist()
total_inst = len(instruments)
print(f"Valid rows: {len(df_valid):,} | Instruments: {total_inst}")

t0 = time.time()
parts = []
for k, (inst, g) in enumerate(df_valid.groupby("instrument_id", sort=False), start=1):
    parts.append(process_one_instrument(g))
    if k % 10 == 0 or k == total_inst:
        done_rows = sum(len(p) for p in parts)
        print(f"  [{k}/{total_inst}] rows ~{done_rows:,} | {time.time()-t0:.1f}s")

df = pd.concat(parts + [df_invalid], axis=0).sort_index()
df = df.drop(columns=["priority_rank"], errors="ignore")
print(f"\nQueue simulation complete. Shape: {df.shape}")

Valid rows: 350,000 | Instruments: 30
  [10/30] rows ~108,995 | 0.5s
  [20/30] rows ~229,639 | 1.1s
  [30/30] rows ~350,000 | 1.6s

Queue simulation complete. Shape: (350000, 17)


## 7. Feature Engineering

We now engineer additional features that capture **temporal patterns**, **congestion effects**, and **instrument reliability trends**. These features are critical because:

- **Temporal features** capture peak-hour and weekend effects on lab congestion
- **Stress index** quantifies how workload and occupancy interact to create bottlenecks
- **Cumulative usage** tracks instrument degradation over the year
- **Rolling failure rate** detects instruments trending toward breakdown
- **Duration-to-queue ratio** captures how long an experiment will "block" others

### 7a. Temporal Features (from booking_time)

In [7]:
# =========================================================
# Temporal features
# =========================================================
df["hour_of_day"] = df["booking_time"].dt.hour.fillna(9).astype(int)
df["day_of_week"] = df["booking_time"].dt.dayofweek.fillna(0).astype(int)  # 0=Mon, 6=Sun
df["is_weekend"]  = (df["day_of_week"] >= 5).astype(int)

# Days since start of observation period (temporal drift proxy)
bt0 = df["booking_time"].min()
df["days_since_start"] = (
    (df["booking_time"] - bt0).dt.total_seconds() / 86400.0
).fillna(0.0)

print("Temporal features added:")
print(f"  hour_of_day range: {df['hour_of_day'].min()} – {df['hour_of_day'].max()}")
print(f"  day_of_week range: {df['day_of_week'].min()} – {df['day_of_week'].max()}")
print(f"  is_weekend mean:   {df['is_weekend'].mean():.3f}")
print(f"  days_since_start:  {df['days_since_start'].min():.0f} – {df['days_since_start'].max():.0f}")

Temporal features added:
  hour_of_day range: 0 – 23
  day_of_week range: 0 – 6
  is_weekend mean:   0.056
  days_since_start:  0 – 365


### 7b. Congestion / Stress Features

In [8]:
# =========================================================
# Stress index = workload × occupancy (lab congestion pressure)
# =========================================================
df["scientist_workload"]   = pd.to_numeric(df.get("scientist_workload", 0), errors="coerce").fillna(0.0)
df["lab_occupancy_level"]  = pd.to_numeric(df.get("lab_occupancy_level", 0), errors="coerce").fillna(0.0)
df["stress_index"] = df["scientist_workload"] * df["lab_occupancy_level"]

# Duration to queue ratio: how much relative congestion this experiment faces
df["duration_to_queue_ratio"] = df["expected_duration"] / (df["queue_length"] + 1)

print("Congestion features added:")
print(f"  stress_index median:            {df['stress_index'].median():.1f}")
print(f"  duration_to_queue_ratio median: {df['duration_to_queue_ratio'].median():.2f}")

Congestion features added:
  stress_index median:            282.0
  duration_to_queue_ratio median: 0.07


### 7c. Instrument Reliability Features

These features capture how much an instrument has been used and its recent track record.  
We compute them **per instrument, sorted by booking time** to avoid data leakage.

In [9]:
# =========================================================
# Instrument cumulative usage (hours) — degradation proxy
# =========================================================
df = df.sort_values("booking_time").reset_index(drop=True)

df["instrument_cumulative_hours"] = (
    df.groupby("instrument_id")["expected_duration"]
    .cumsum() / 60.0  # convert minutes → hours
)

# =========================================================
# Instrument rolling failure rate (last 30 days by booking_time)
# =========================================================
df["machine_failure"] = pd.to_numeric(df.get("machine_failure", 0), errors="coerce").fillna(0).astype(int)

def rolling_failure_rate(group, window_days=30):
    """For each row, compute the failure rate among previous experiments
    on the same instrument within the last `window_days` days."""
    group = group.sort_values("booking_time")
    times = group["booking_time"].values
    fails = group["machine_failure"].values
    n = len(group)
    rates = np.zeros(n, dtype=float)
    window_ns = np.timedelta64(window_days, 'D').astype('int64')
    
    left = 0
    cum_fail = 0
    for i in range(n):
        # Expand window — include current row's failure in running total
        cum_fail += fails[i]
        # Shrink left pointer: exclude rows older than window_days before current
        curr_ns = times[i].astype('int64')
        while left < i and (curr_ns - times[left].astype('int64')) > window_ns:
            cum_fail -= fails[left]
            left += 1
        count = i - left + 1
        rates[i] = cum_fail / count if count > 0 else 0.0
    
    return pd.Series(rates, index=group.index)

print("Computing rolling 30-day failure rate per instrument...", flush=True)
t0 = time.time()
df["instrument_recent_failure_rate"] = (
    df.groupby("instrument_id", group_keys=False)
    .apply(rolling_failure_rate)
)
print(f"  Done in {time.time()-t0:.1f}s")

print(f"\nReliability features added:")
print(f"  instrument_cumulative_hours median:      {df['instrument_cumulative_hours'].median():.0f}")
print(f"  instrument_recent_failure_rate mean:     {df['instrument_recent_failure_rate'].mean():.4f}")

Computing rolling 30-day failure rate per instrument...
  Done in 0.5s

Reliability features added:
  instrument_cumulative_hours median:      5720
  instrument_recent_failure_rate mean:     0.0404


/var/folders/80/vx7bsx9917v8fw_1j_dc367h0000gn/T/ipykernel_77360/2530992428.py:45: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(rolling_failure_rate)


## 8. Summary & Save

In [10]:
# =========================================================
# Show all new columns
# =========================================================
new_cols = [
    "priority", "queue_wait_min", "queue_length",
    "hour_of_day", "day_of_week", "is_weekend", "days_since_start",
    "stress_index", "duration_to_queue_ratio",
    "instrument_cumulative_hours", "instrument_recent_failure_rate"
]

print(f"=== Enriched Dataset ===")
print(f"Shape: {df.shape}")
print(f"\nNew/engineered columns ({len(new_cols)}):")
for c in new_cols:
    if c in df.columns:
        print(f"  ✓ {c:40s}  dtype={df[c].dtype}  nulls={df[c].isna().sum()}")
    else:
        print(f"  ✗ {c} — MISSING")

print(f"\nOriginal workflow columns: {len(df.columns) - len(new_cols)}")
print(f"Total columns: {len(df.columns)}")

=== Enriched Dataset ===
Shape: (350000, 25)

New/engineered columns (11):
  ✓ priority                                  dtype=object  nulls=0
  ✓ queue_wait_min                            dtype=float64  nulls=0
  ✓ queue_length                              dtype=int64  nulls=0
  ✓ hour_of_day                               dtype=int64  nulls=0
  ✓ day_of_week                               dtype=int64  nulls=0
  ✓ is_weekend                                dtype=int64  nulls=0
  ✓ days_since_start                          dtype=float64  nulls=0
  ✓ stress_index                              dtype=int64  nulls=0
  ✓ duration_to_queue_ratio                   dtype=float64  nulls=0
  ✓ instrument_cumulative_hours               dtype=float64  nulls=0
  ✓ instrument_recent_failure_rate            dtype=float64  nulls=0

Original workflow columns: 14
Total columns: 25


In [11]:
# Save enriched dataset
df.to_csv(OUT_PATH, index=False)
print(f"[saved] {OUT_PATH.resolve()}")
print(f"Rows: {len(df):,} | Cols: {len(df.columns)}")

# Quick sanity check
df[["instrument_id", "priority", "queue_length", "queue_wait_min",
    "hour_of_day", "stress_index", "instrument_cumulative_hours",
    "instrument_recent_failure_rate"]].head(10)

[saved] /Users/marianasaca/Downloads/roche-capstone/data/enriched/workflow_logs_priority_queue.csv
Rows: 350,000 | Cols: 25


,instrument_id,priority,queue_length,queue_wait_min,hour_of_day,stress_index,instrument_cumulative_hours,instrument_recent_failure_rate
0,Incubator_04,High,0,0.0,0,177,1.50,0.0
1,Spectrometer_04,Normal,0,0.0,0,216,0.75,0.0
2,HPLC_03,Normal,0,0.0,1,43,0.75,0.0
3,Spectrometer_01,Normal,0,0.0,1,255,0.75,0.0
4,PCR_00,Normal,0,0.0,1,486,0.50,0.0
5,PCR_03,Normal,0,0.0,2,308,0.50,0.0
6,Microscope_04,Normal,0,0.0,2,88,1.00,0.0
7,Incubator_04,High,0,0.0,2,225,3.00,0.0
8,PCR_00,High,0,0.0,4,141,2.00,0.0
9,Microscope_00,Normal,0,0.0,4,212,1.00,0.0
